# ✈️ Extracción y Procesamiento Inicial de Datos de Tráfico Aéreo (OpenSky)

La disponibilidad de datos aeronáuticos en tiempo real permite analizar patrones de movilidad aérea, comportamiento de aeronaves y variaciones operativas a escala global.  
Este proyecto desarrolla un **pipeline ETL** que ingesta información del endpoint público de **OpenSky Network**, centrado en capturar el estado actual de miles de vuelos activos en simultáneo.

La API proporciona variables clave como:

- identificador **ICAO24**  
- país de origen  
- latitud / longitud  
- altitudes barométrica y geométrica  
- velocidad, rumbo, tasa vertical  
- estado en tierra o en vuelo  
- timestamp del servidor  

Estos datos se transforman y almacenan en un **Data Lake local** siguiendo la arquitectura **Bronze → Silver → Gold**, lo que permite realizar análisis históricos, construir métricas aeronáuticas y preparar la futura migración a entornos de nube (Azure).

## Objetivos

**Extracción (Bronze):**
- Consumir el endpoint `states/all` de OpenSky.  
- Normalizar la estructura JSON y convertirla en tabla.  
- Incorporar timestamps (servidor y extracción).  
- Guardar los datos crudos en **Delta Lake**.

**Transformación (Silver):**
- Limpiar valores faltantes y tipos de datos.  
- Estandarizar columnas y coordenadas.  
- Preparar la tabla para análisis temporal.

**Métricas (Gold):**
- Crear features de movilidad aérea: altitud efectiva, variación de velocidad, indicadores de vuelo/estacionamiento, etc.  
- Generar datasets optimizados para visualización y análisis exploratorio.

## Alcance y supuestos

- Se utilizan exclusivamente datos públicos provistos por OpenSky Network.  
- La extracción se realiza bajo límites de la API pública (sin autenticación obligatoria).  
- El objetivo es **práctico y educativo**, orientado al portfolio de Ingeniería de Datos.

## Reproducibilidad

- Dependencias detalladas en `requirements.txt`.  
- Las rutas del Data Lake se configuran en `pipeline.conf`.  
- Todas las funciones auxiliares se encuentran en `src/etl_utils.py`.

---

**Estructura del notebook:**

0) Configuración inicial  
1) Extracción del endpoint `states/all`  
2) Normalización del JSON  
3) Limpieza y estandarización mínima  
4) Almacenamiento en Delta Lake (capa Bronze)  
5) Verificación y vista preliminar de los datos  

## 0. Configuración inicial

En este paso se importan todas las librerías necesarias y las funciones auxiliares definidas en `etl_utils.py`.  
Este enfoque permite mantener el notebook **ordenado, modular y fácilmente reproducible**, centralizando en un único módulo las operaciones comunes del pipeline ETL: extracción desde la API pública de **OpenSky Network**, normalización del JSON, estandarización de columnas y escritura en las distintas capas del **Data Lake local** (Bronze → Silver → Gold).

El objetivo de esta sección es garantizar que todas las dependencias estén correctamente cargadas antes de iniciar el proceso de extracción y almacenamiento.

In [2]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

# Importar funciones auxiliares
from etl_utils import *

# Librerías comunes
import pandas as pd

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


## 1. Autenticación y lectura de configuración

La configuración del proyecto se administra mediante el archivo `pipeline.conf`,  
que centraliza parámetros como:
- la **URL base** de la API de OpenSky Network  
- credenciales opcionales para *Basic Auth* (en caso de usarse)  
- rutas del **Data Lake local**

Aunque la API pública de OpenSky no requiere autenticación obligatoria,almacenar parámetros en un archivo de configuración permite:
- mantener el notebook limpio  
- evitar credenciales expuestas en el código  
- facilitar la migración futura a servicios en la nube (Azure Key Vault)

El archivo se lee mediante `ConfigParser`, lo que permite obtener los valores  
en forma segura y reusable.


In [3]:
# Se instancia el parser y se lee el archivo de configuración
from configparser import ConfigParser

parser = ConfigParser()
parser.read("../pipeline.conf")

['../pipeline.conf']

In [4]:
# Parámetros de conexión
api_config = parser["api-opensky"]
base_url = api_config["base_url"]

In [5]:
print("📄 Configuración cargada correctamente.")
print(f"URL base: {base_url}")

📄 Configuración cargada correctamente.
URL base: https://opensky-network.org/api/states/all


In [6]:
# Prueba de conexión a la API OpenSky
response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    print(f"La petición fue exitosa. Tipo de respuesta: {type(data)}")

    # Claves principales del JSON
    print(f"Claves principales recibidas: {list(data.keys())[:5]}")

    # Inspección parcial de 'states'
    print("\nPrimeras 2 aeronaves registradas:")
    pprint(data["states"][:2])

else:
    print(f"❌ Error en la petición: {response.status_code}, {response.content}")
print("✅ Prueba de conexión a la API realizada.")

La petición fue exitosa. Tipo de respuesta: <class 'dict'>
Claves principales recibidas: ['time', 'states']

Primeras 2 aeronaves registradas:
[['39de4f',
  'TVF76QA ',
  'France',
  1763206177,
  1763206184,
  0.9783,
  48.6191,
  7696.2,
  False,
  207.14,
  232.16,
  9.1,
  None,
  7719.06,
  '7676',
  False,
  0],
 ['39de4e',
  'TVF18ZQ ',
  'France',
  1763206183,
  1763206183,
  1.3229,
  47.8852,
  5234.94,
  False,
  188.28,
  24.88,
  -9.75,
  None,
  5273.04,
  '1000',
  False,
  0]]
✅ Prueba de conexión a la API realizada.


## 2. Capa Bronze — Extracción y almacenamiento de datos crudos

En esta etapa se realiza la **extracción directa de datos** desde la API pública de **OpenSky Network**, que provee información en tiempo real sobre aeronaves detectadas a nivel global.  
El objetivo es obtener el conjunto completo de registros tal como es devuelto por el endpoint `states/all` y conservarlo en su forma más fiel dentro de la capa **🟤 Bronze** del Data Lake.

Los datos obtenidos incluyen:

- Identificador único **ICAO24**  
- Indicativo de llamada (**callsign**)  
- País de origen  
- Posición geográfica (latitud, longitud)  
- Altitudes barométrica y geométrica  
- Velocidad, rumbo y tasa vertical  
- Estado operativo (en tierra o en vuelo)  
- Timestamp del servidor (`time`) correspondiente a la captura

Cada aeronave es representada inicialmente como una lista ordenada de 17 elementos, por lo que en esta etapa se prioriza **preservar los datos crudos** antes de aplicar procesos de estructuración o limpieza.

Los datos se almacenan en **formato Delta Lake**, dentro del directorio:

`data/etl_datalake/bronze/api_opensky/`

empleando el modo **`overwrite`**, ya que la información corresponde a un snapshot puntual del estado global del tráfico aéreo y puede reemplazarse completamente en cada actualización.


### 2.1 Extracción de datos estáticos

Se realiza una *ingesta full* sobre el recurso estático `aircraftDatabase.csv` provisto por **OpenSky Network**, que contiene información descriptiva sobre aeronaves (modelo, fabricante, typecode, operador, entre otros metadatos relevantes).

Si bien este dataset no proviene de un endpoint API, constituye la **fuente oficial de referencia** publicada por la plataforma.  
Dado que cambia muy poco en el tiempo, se incorpora en su totalidad en cada ejecución y se sobrescribe la versión previa (*mode="overwrite"*).

Este enfoque evita duplicados, simplifica el pipeline y garantiza la disponibilidad de la versión más reciente.

Los datos se almacenan en la capa 🟤 **Bronze** en formato **Delta Lake**, preservados tal como fueron obtenidos.

In [7]:
# --- Extracción de datos estáticos — OpenSky aircraft metadata ---

# URL del recurso estático oficial
metadata_url = "https://opensky-network.org/datasets/metadata/aircraftDatabase.csv"

# Descarga del archivo CSV
# Nota: Estos metadatos cambian muy poco, por lo que se aplicará ingesta "full"
df_aircraft = pd.read_csv(metadata_url)

# Vista preliminar del dataset cargado
print("📄 Dataset de metadatos cargado correctamente.")
print(f"Filas: {df_aircraft.shape[0]} | Columnas:, {df_aircraft.shape[1]}")
df_aircraft.head(3)

📄 Dataset de metadatos cargado correctamente.
Filas: 520000 | Columnas:, 27


,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,NaN,L1P,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,NaN,L2P,NaN,...,NaN,1977-01-01,NaN,NaN,LYCOMING TI0-540 SER,False,False,False,NaN,NaN


### 2.2 Extracción de datos dinámicos

Para los datos dinámicos se utiliza la información proveniente del endpoint `states/all`, que ofrece el estado en tiempo real de miles de aeronaves a nivel global.  
Dado que estos datos se actualizan continuamente, cada ejecución captura un *snapshot* independiente del tráfico aéreo del momento.

En este caso no se aplica una actualización incremental, ya que el conjunto de aeronaves presentes varía en cada consulta y no existe un identificador temporal que permita rastrear cambios de manera estricta.  
Por ello, cada snapshot se almacena íntegramente en la capa 🟤 *Bronze*, preservando la evolución temporal entre ejecuciones y habilitando análisis posteriores en Silver y Gold.

In [8]:
# --- Extracción de datos dinámicos — OpenSky states/all ---

# Obtención del snapshot dinámico mediante la función auxiliar
json_data = get_opensky_states()

# Vista preliminar del JSON recibido
print("🔑 Claves principales:", list(json_data.keys()))
print("🛫 Cantidad de aeronaves detectadas:", len(json_data["states"]))

# Visualización de los primeros registros crudos
print("\nPrimeras 2 aeronaves (formato crudo):")
pprint(json_data["states"][:2])

🔑 Claves principales: ['time', 'states']
🛫 Cantidad de aeronaves detectadas: 6235

Primeras 2 aeronaves (formato crudo):
[['39de4f',
  'TVF76QA ',
  'France',
  1763206177,
  1763206184,
  0.9783,
  48.6191,
  7688.58,
  False,
  207.14,
  232.16,
  9.1,
  None,
  7711.44,
  '7676',
  False,
  0],
 ['39de4e',
  'TVF18ZQ ',
  'France',
  1763206191,
  1763206194,
  1.3293,
  47.8944,
  5173.98,
  False,
  186.66,
  24.94,
  -10.08,
  None,
  5189.22,
  '1000',
  False,
  0]]


### 2.3 Guardado en Bronze — Delta Lake

Una vez realizada la extracción de los datos estáticos y dinámicos, ambos recursos se almacenan en la capa 🟤 *Bronze* del Data Lake en formato **Delta Lake**.  
En esta etapa no se aplican transformaciones ni procesos de limpieza: los datos se preservan tal como fueron obtenidos desde la API, cumpliendo el propósito de Bronze como zona de almacenamiento crudo.

Los metadatos estáticos se guardan mediante una *ingesta full* (mode="overwrite"), ya que su contenido cambia muy poco y resulta más simple reemplazar el dataset completo en cada ejecución.

Los datos dinámicos provenientes de `states/all` se guardan como snapshots independientes, manteniendo la trazabilidad temporal de cada captura.  
Este enfoque permite conservar el historial de estados del tráfico aéreo para posteriores análisis en las capas Silver y Gold.

Las rutas de salida se definen dentro del directorio:

`data/etl_datalake/bronze/`

utilizando subdirectorios separados para **datos estáticos** y **datos dinámicos**.

In [9]:
# --- Definición de rutas del Data Lake (Bronze) ---

# Se mantiene fuera de la carpeta notebooks para centralizar los datos.
datalake_root = "../data/etl_datalake"

# Carpeta raíz del dominio OpenSky dentro de Bronze
bronze_dir = f"{datalake_root}/bronze/api_opensky"

# Subcarpetas descriptivas según el tipo de recurso
static_dir  = f"{bronze_dir}/aircraft_metadata"
dynamic_dir = f"{bronze_dir}/states"

print("📁 Rutas definidas:")
print("Static  →", static_dir)
print("Dynamic →", dynamic_dir)

📁 Rutas definidas:
Static  → ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
Dynamic → ../data/etl_datalake/bronze/api_opensky/states


In [10]:
# --- Guardado en Bronze — Metadatos estáticos (Delta Lake) ---

# Ingesta full: se sobrescribe el dataset completo en cada ejecución,
# dado que los metadatos de aeronaves cambian muy poco en el tiempo.
save_data_as_delta(
    df=df_aircraft,
    path=static_dir,
    mode="overwrite"
)

print("🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).")

💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).


In [11]:
# --- Guardado en Bronze — Snapshot dinámico (Delta Lake) ---

# Conversión a DataFrame del recurso dinámico "states"
df_dynamic = pd.DataFrame(json_data["states"])

# En algunos snapshots, ciertas columnas pueden venir completamente vacías.
# Delta Lake no acepta columnas de tipo "Null" (100% None), por lo que se eliminan
# únicamente aquellas que no contienen ningún valor real.
df_dynamic = df_dynamic.dropna(axis=1, how="all")

print("📄 Shape del snapshot después de eliminar columnas vacías:", df_dynamic.shape)

# Guardado incremental (append) para preservar el historial de snapshots.
save_data_as_delta(
    df=df_dynamic,
    path=dynamic_dir,
    mode="append"
)

print("🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).")

📄 Shape del snapshot después de eliminar columnas vacías: (6235, 16)
💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/states
🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).


## 3. Capa Silver — Normalización y limpieza

La capa **Silver** aplica transformaciones sobre los datos almacenados en Bronze con el objetivo de obtener un conjunto de datos limpio, tabular y estructurado, adecuado para análisis posteriores.

En esta etapa se realizan:
- renombrado de columnas  
- estandarización de tipos  
- conversión de timestamps  
- selección de atributos relevantes  
- reducción de nulos  
- enriquecimiento opcional mediante cruce entre datasets

El procesamiento se organiza en dos pasos:

### 3.1 Metadatos estáticos (aircraft_metadata)
Depuración de columnas, estandarización de nombres y construcción de la **tabla de referencia de aeronaves**, que funcionará como conjunto de metadatos estable para enriquecer los datos dinámicos.

### 3.2 Datos dinámicos (states/all)
Normalización del snapshot, asignación de nombres descriptivos a las columnas, tipificación y preparación para análisis temporal.

Este enfoque garantiza un esquema consistente y una base sólida para la etapa **Gold**, donde se generarán métricas, indicadores y visualizaciones del tráfico aéreo.

### 3.1 Metadatos estáticos — Construcción de la tabla de referencia de aeronaves

El dataset de metadatos de aeronaves obtenido desde OpenSky contiene información descriptiva asociada a cada código `icao24`.  
Aunque estos datos cambian poco en el tiempo, llegan con numerosas columnas incompletas, valores inconsistentes y atributos que no aportan información útil.

En esta etapa se construye una **tabla de referencia de aeronaves** limpia y estable mediante los siguientes pasos:

- **eliminación de columnas con nulos al 100%**
- **verificación y eliminación de duplicados** en la clave primaria `icao24`
- **detección y eliminación de columnas sin variabilidad** (valores constantes)
- **selección de atributos relevantes**, conservando únicamente información descriptiva útil:
  - `icao24`
  - `registration`
  - `manufacturername`
  - `model`
  - `typecode`
  - `owner`
  - `built`
  - `engines`
- **tipificación suave**, asegurando que todos los campos descriptivos estén representados como `string`

El resultado es una tabla compacta, consistente y adecuada para enriquecer los datos dinámicos procesados en la siguiente etapa.

In [12]:
# Se lee Bronze (Delta)
dt_aircraft_metadata = DeltaTable(static_dir)

In [13]:
# Se convierte a Pandas para trabajar en las transformaciones
dt_aircraft_metadata_bronze = dt_aircraft_metadata.to_pandas()

In [14]:
# Se crea una copia para aplicar las transformaciones
df_aircraft_cleaned  = dt_aircraft_metadata_bronze.copy()

In [15]:
# Vista preliminar
df_aircraft_cleaned.head(3)

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,None,None,None,None,None,None,None,None,None,None,...,NaN,None,None,NaN,None,False,False,False,None,None
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,None,L1P,None,...,NaN,None,None,NaN,None,False,False,False,None,None
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,None,L2P,None,...,NaN,1977-01-01,None,NaN,LYCOMING TI0-540 SER,False,False,False,None,None


In [16]:
# --- Inspección inicial del dataset estático 
df_aircraft_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520000 entries, 0 to 519999
Data columns (total 27 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   icao24               519999 non-null  object 
 1   registration         516526 non-null  object 
 2   manufacturericao     428801 non-null  object 
 3   manufacturername     438566 non-null  object 
 4   model                440426 non-null  object 
 5   typecode             479947 non-null  object 
 6   serialnumber         437118 non-null  object 
 7   linenumber           971 non-null     object 
 8   icaoaircrafttype     428784 non-null  object 
 9   operator             23698 non-null   object 
 10  operatorcallsign     40090 non-null   object 
 11  operatoricao         41402 non-null   object 
 12  operatoriata         8066 non-null    object 
 13  owner                435913 non-null  object 
 14  testreg              297 non-null     object 
 15  registered       

In [17]:
# --- Verificación de valores nulos en la clave primaria (icao24) ---
nulos = df_aircraft_cleaned['icao24'].isna().sum()
print(f"Cantidad de valores nulos en 'icao24': {nulos}")

Cantidad de valores nulos en 'icao24': 1


In [18]:
# --- Verificación de valores vacíos o espacios en 'icao24' ---
vacios = (df_aircraft_cleaned['icao24'].astype(str).str.strip() == "").sum()
print(f"Cantidad de valores vacíos en 'icao24': {vacios}")

Cantidad de valores vacíos en 'icao24': 0


In [19]:
# --- Verificación de duplicados en 'icao24' ---
duplicados = df_aircraft_cleaned['icao24'].duplicated().sum()
print(f"Cantidad de valores duplicados en 'icao24': {duplicados}")

Cantidad de valores duplicados en 'icao24': 2


In [20]:
# --- Inspección de los valores duplicados en 'icao24' ---

# Filtramos únicamente las filas cuyo icao24 aparece más de una vez
df_aircraft_cleaned[df_aircraft_cleaned['icao24'].duplicated(keep=False)].sort_values('icao24')

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
31226,ae690b,710376,PANHA,None,None,B06,None,None,H1T,None,...,NaN,None,None,NaN,None,False,False,False,None,None
423031,ae690b,710376,PANHA,None,None,B06,None,None,H1T,None,...,NaN,None,None,NaN,None,False,False,False,None,None
82947,ae6963,170000,LOCKHEED MARTIN,Lockheed,C-130J Hercules C.5,C30J,5483,None,L4T,United States Navy,...,NaN,None,None,NaN,None,False,False,False,"2020 July: ""Blue Angels"" Team Support Aircaft,...",Reserved
302446,ae6963,170000,LOCKHEED MARTIN,Lockheed,C-130J Hercules C.5,C30J,5483,None,L4T,United States Navy,...,NaN,None,None,NaN,None,False,False,False,"2020 July: ""Blue Angels"" Team Support Aircaft,...",Reserved


In [21]:
# --- Eliminación de valores nulos en la clave primaria (icao24) ---
df_aircraft_cleaned = df_aircraft_cleaned[df_aircraft_cleaned['icao24'].notna()]

In [22]:
# --- Eliminación de duplicados en la clave primaria (icao24) ---
df_aircraft_cleaned = df_aircraft_cleaned.drop_duplicates(subset='icao24', keep='first')

In [23]:
# --- Verificación post-limpieza ---
print("Nulos en icao24:", df_aircraft_cleaned['icao24'].isna().sum())
print("Duplicados en icao24:", df_aircraft_cleaned['icao24'].duplicated().sum())

Nulos en icao24: 0
Duplicados en icao24: 0


In [24]:
    # --- Estandarización de nombres de columnas (Silver) ---
# Se normalizan todos los nombres de columnas para garantizar un esquema consistente.
# Este formato (snake_case + minúsculas) es el estándar utilizado en pipelines ETL
# y evita ambigüedades en transformaciones posteriores y uniones entre datasets.

df_aircraft_cleaned.columns = (
    df_aircraft_cleaned.columns
        .str.lower()            # forzar minúsculas
        .str.strip()            # eliminar espacios al inicio y final
        .str.replace(r"\s+", "_", regex=True)   # reemplazar espacios por '_'
        .str.replace(r"[^a-z0-9_]", "", regex=True)  # eliminar caracteres no válidos
)
df_aircraft_cleaned.head(3)

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categorydescription
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,None,L1P,None,...,NaN,None,None,NaN,None,False,False,False,None,None
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,None,L2P,None,...,NaN,1977-01-01,None,NaN,LYCOMING TI0-540 SER,False,False,False,None,None
3,a7a809,N5926K,ROCKWELL,None,None,AC90,None,None,L2T,None,...,NaN,None,None,NaN,None,False,False,False,None,None


In [25]:
# --- Eliminación de columnas con 100% de valores nulos ---
# En esta etapa se descartan las columnas cuyo contenido es completamente nulo.
# Estas variables no aportan información a la tabla de referencia de aeronaves y
# solo generarían ruido en las siguientes etapas (joins, estadísticas, almacenamiento).

null_pct = df_aircraft_cleaned.isna().mean() * 100
cols_to_drop = null_pct[null_pct == 100].index.tolist()

df_aircraft_cleaned = df_aircraft_cleaned.drop(columns=cols_to_drop)

print(f"📉 Columnas eliminadas por nulos al 100%: {cols_to_drop}")
print(f"📦 Total de columnas restantes: {df_aircraft_cleaned.shape[1]}")

📉 Columnas eliminadas por nulos al 100%: ['status', 'seatconfiguration']
📦 Total de columnas restantes: 25


In [26]:
# --- Definición de columnas candidatas según documentación de OpenSky ---

# Atributos principales esperados en la tabla de referencia de aeronaves
cols_principales = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "operator"
]

# Atributos complementarios (útiles si contienen variabilidad)
cols_complementarias = [
    "owner",
    "built",
    "engines",
    "modes",
    "adsb",
    "acars"
]

# Combinación de columnas candidatas
cols_candidatas = cols_principales + cols_complementarias

# Se filtran solo las columnas presentes en el dataset
cols_candidatas = [c for c in cols_candidatas if c in df_aircraft_cleaned.columns]

print("📌 Columnas candidatas detectadas:", cols_candidatas)


📌 Columnas candidatas detectadas: ['icao24', 'registration', 'manufacturername', 'model', 'typecode', 'operator', 'owner', 'built', 'engines', 'modes', 'adsb', 'acars']


In [27]:
# --- Verificación de variabilidad para cada columna candidata ---

print("🔎 Verificación de variabilidad en columnas candidatas:\n")

cols_constantes = []

for col in cols_candidatas:
    valores_unicos = df_aircraft_cleaned[col].dropna().unique()
    n_unicos = len(valores_unicos)

    print(f"{col}: {n_unicos} valores únicos")

    # Si la columna tiene 0 o 1 valores distintos → no aporta información
    if n_unicos <= 1:
        cols_constantes.append(col)

print("\n⚠️ Columnas con variabilidad nula:", cols_constantes)

🔎 Verificación de variabilidad en columnas candidatas:

icao24: 519997 valores únicos
registration: 514254 valores únicos
manufacturername: 41540 valores únicos
model: 34482 valores únicos
typecode: 1933 valores únicos
operator: 4001 valores únicos
owner: 224893 valores únicos
built: 381 valores únicos
engines: 7529 valores únicos
modes: 1 valores únicos
adsb: 1 valores únicos
acars: 1 valores únicos

⚠️ Columnas con variabilidad nula: ['modes', 'adsb', 'acars']


In [28]:
# --- Eliminación de columnas sin información útil ---

df_aircraft_cleaned.drop(columns=cols_constantes, inplace=True)

print("🧹 Columnas eliminadas:", cols_constantes)
print("📦 Columnas restantes:", df_aircraft_cleaned.columns.tolist())

🧹 Columnas eliminadas: ['modes', 'adsb', 'acars']
📦 Columnas restantes: ['icao24', 'registration', 'manufacturericao', 'manufacturername', 'model', 'typecode', 'serialnumber', 'linenumber', 'icaoaircrafttype', 'operator', 'operatorcallsign', 'operatoricao', 'operatoriata', 'owner', 'testreg', 'registered', 'reguntil', 'built', 'firstflightdate', 'engines', 'notes', 'categorydescription']


In [29]:
# --- Selección final de atributos relevantes para la tabla de referencia de aeronaves ---

cols_finales = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "owner",
    "built",
    "engines"
]

# Verificación final: se mantienen solo las columnas presentes tras la depuración
cols_finales = [c for c in cols_finales if c in df_aircraft_cleaned.columns]

# Construcción de la tabla de referencia (versión Silver)
df_aircraft_cleaned = df_aircraft_cleaned[cols_finales]

print("📘 Tabla de referencia de aeronaves — Silver:")
df_aircraft_cleaned.head(5)

📘 Tabla de referencia de aeronaves — Silver:


,icao24,registration,manufacturername,model,typecode,owner,built,engines
1,aa3487,N757F,Raytheon Aircraft Company,A36,BE36,Vintage Aircraft Llc,None,None
2,a4fa61,N42MH,Piper,PA-31-350,PA31,Tvpx Aircraft Solutions Inc Trustee,1977-01-01,LYCOMING TI0-540 SER
3,a7a809,N5926K,None,None,AC90,None,None,None
4,391927,F-GGJH,Robin,DR.400 160 Chevalier,DR40,Private,None,None
5,503c21,LY-KNA,Impulse Aircraft,Impulse 100,ZZZZ,Private,None,None


In [30]:
# --- Verificación inicial de tipos de datos ---
# Se revisa el tipo actual de cada columna antes de la tipificación suave.
df_aircraft_cleaned.dtypes

icao24              object
registration        object
manufacturername    object
model               object
typecode            object
owner               object
built               object
engines             object
dtype: object

In [31]:
# --- Tipificación suave: asegurar tipo string en columnas descriptivas ---
# Se aplica después de la depuración y selección de columnas finales.

cols_string = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "owner",
    "built",
    "engines"
]

# Conversión a string (astype evita problemas con valores mixtos)
df_aircraft_cleaned[cols_string] = df_aircraft_cleaned[cols_string].astype("string")

print("🔤 Tipificación suave aplicada (todas las columnas relevantes como string).")
df_aircraft_cleaned.dtypes

🔤 Tipificación suave aplicada (todas las columnas relevantes como string).


icao24              string[python]
registration        string[python]
manufacturername    string[python]
model               string[python]
typecode            string[python]
owner               string[python]
built               string[python]
engines             string[python]
dtype: object

### 3.2 Datos dinámicos — Normalización del snapshot de estados (states/all)

El endpoint `states/all` de OpenSky proporciona un **snapshot en tiempo real** del estado de todas las aeronaves detectadas por la red ADS-B en el instante de la consulta.  
A diferencia de los metadatos estáticos, estos datos son **altamente volátiles**, se actualizan continuamente y contienen información esencial para el análisis del tráfico aéreo:

- posición (`longitude`, `latitude`)
- altitud (`baro_altitude`)
- velocidad (`velocity`)
- rumbo (`true_track`)
- tasa de ascenso/descenso (`vertical_rate`)
- estado del transpondedor
- timestamp del servidor y del sensor

Sin embargo, este snapshot llega en forma de **listas anónimas**, donde cada registro es un arreglo ordenado y no un diccionario.  
Esto hace que la primera tarea del pipeline sea convertir estos datos en un **DataFrame tabular con nombres de columnas claros y estandarizados**.

En esta etapa se realizarán las siguientes transformaciones:

- asignación de nombres descriptivos a cada columna según la documentación oficial  
- conversión de tipos (float, int, booleanos)  
- normalización de valores faltantes y no válidos  
- conversión de timestamps a formato datetime  
- creación de columnas temporales derivadas (fecha, hora, minuto)  
- preparación del esquema final para particionado temporal en Delta Lake  

El objetivo es lograr un dataset dinámico **limpio, tipado y analíticamente consistente**, listo para integrarse con la tabla de referencia de aeronaves y para almacenarse en Silver de forma particionada por timestamp.

### 3.2.1 Lectura del último snapshot dinámico desde Bronze

Los datos dinámicos del endpoint `states/all` se almacenan en Bronze como **snapshots independientes**, cada uno correspondiente al momento exacto en que fue ejecutada la ingesta.

Para iniciar el procesamiento Silver, se carga **el snapshot más reciente**, que representa el estado actual del tráfico aéreo en el momento de ejecución.  
Este enfoque permite trabajar siempre con la versión más actualizada sin perder el historial completo almacenado en Bronze.

La lectura se realiza directamente desde la tabla Delta Lake ubicada en:

`data/etl_datalake/bronze/api_opensky/states/`

Una vez cargado el dataset, se convierte a **Pandas** para aplicar las transformaciones de limpieza y normalización necesarias para la etapa Silver.

In [32]:
# --- Lectura del snapshot más reciente desde Bronze (Delta Lake) ---
dt_states = DeltaTable(dynamic_dir)

In [33]:
# Se convierte a Pandas para trabajar en las transformaciones
df_states_bronze = dt_states.to_pandas()

In [34]:
# Se crea una copia para aplicar las transformaciones
df_states_cleaned  = df_states_bronze.copy()

In [35]:
# Vista preliminar
df_states_cleaned.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,13,14,15,16
0,39de4f,TVF76QA,France,1.763206e+09,1763206184,0.9783,48.6191,7688.58,False,207.14,232.16,9.10,7711.44,7676,False,0
1,39de4e,TVF18ZQ,France,1.763206e+09,1763206194,1.3293,47.8944,5173.98,False,186.66,24.94,-10.08,5189.22,1000,False,0
2,39de4b,TVF87NV,France,1.763206e+09,1763206194,2.0260,38.2462,11582.40,False,253.31,36.66,0.00,11841.48,5571,False,0


### 3.2.2 Normalización del snapshot dinámico

El endpoint `states/all` devuelve cada aeronave como una lista ordenada de 17 elementos cuya posición corresponde a un atributo específico (posición, altitud, velocidad, estado del transpondedor, etc.).  
Al ingresar estos datos en Pandas, las columnas aparecen indexadas numéricamente (`0`, `1`, `2`, …), por lo que el primer paso es asignarles **nombres descriptivos** basados en la documentación oficial de OpenSky Network.

Una vez renombradas, se aplican las transformaciones básicas necesarias para preparar el snapshot dinámico en formato tabular:

- **tipificación de columnas** numéricas, booleanas y de texto  
- **conversión de timestamps** (`time_position`, `last_contact`) a formato datetime  
- **reducción de columnas sin información**  
- **normalización general** para garantizar consistencia del esquema

Este proceso deja el snapshot dinámico en un formato limpio y estructurado, apto para análisis temporal y para su posterior integración con la tabla de referencia de aeronaves en etapas Silver y Gold.

In [36]:
# --- Renombrado de columnas del snapshot dinámico (states/all) ---

column_names = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

df_states_cleaned.columns = column_names

print("✅ Columnas renombradas correctamente.")
df_states_cleaned.head(3)


✅ Columnas renombradas correctamente.


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source
0,39de4f,TVF76QA,France,1.763206e+09,1763206184,0.9783,48.6191,7688.58,False,207.14,232.16,9.10,7711.44,7676,False,0
1,39de4e,TVF18ZQ,France,1.763206e+09,1763206194,1.3293,47.8944,5173.98,False,186.66,24.94,-10.08,5189.22,1000,False,0
2,39de4b,TVF87NV,France,1.763206e+09,1763206194,2.0260,38.2462,11582.40,False,253.31,36.66,0.00,11841.48,5571,False,0
